# Sparse AI V3+ Cloud TPU v5e / v6e (Trillium) Execution Notebook
**DMA-Skipping Pallas Block-List Attention (`pallas_block_list_attention`) & Multi-Prime Ultrametric Benchmarks**

### Recommended Frontier Models by Colab TPU Tier:
- **Colab TPU v6e-1 (Trillium, 32 GB HBM)**:
  - `Qwen/Qwen3-8B` (~16 GB `bf16`, 4:1 GQA, `head_dim=128`, QK-Norm)
  - `meta-llama/Meta-Llama-3.1-8B-Instruct` (~16 GB `bf16`, 4:1 GQA)
  - `google/gemma-4-E4B-it` (~16 GB `bf16`, 5:1 Sliding/Global)
- **Colab TPU v5e-1 (16 GB HBM)**:
  - `Qwen/Qwen3-4B` (~8 GB `bf16`, 4:1 GQA, `head_dim=128`, QK-Norm)
  - `meta-llama/Llama-3.2-3B-Instruct` (~6 GB `bf16`, 3:1 GQA)
  - `google/gemma-4-E2B-it` / `google/gemma-3-4b-it`

In [ ]:
# 1. TPU Runtime & HF_TOKEN Secret Initialization
import os
import sys
import subprocess
import importlib

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[Auth] Loaded HF_TOKEN from Colab Secrets.')
except Exception as e:
    print(f'[Auth] Not running in Colab or HF_TOKEN secret not found ({e}); checking environment.')

REPO_URL = os.environ.get('SPARSE_AI_REPO', 'https://github.com/sneed-and-feed/sparse-ai.git')
if not os.path.exists('src/ultrametric_jax') and not os.path.exists('../src/ultrametric_jax'):
    subprocess.run(['git', 'clone', REPO_URL, 'sparse-ai'], check=False)
    if os.path.exists('sparse-ai'):
        os.chdir('sparse-ai')
elif os.path.exists('../src/ultrametric_jax') and not os.path.exists('src/ultrametric_jax'):
    os.chdir('..')

# Always pull latest commits if /content/sparse-ai already existed
if os.path.exists('.git'):
    subprocess.run(['git', 'pull', '--ff-only'], check=False)

sys.path.insert(0, os.path.abspath('src'))

import jax
import jax.numpy as jnp
print(f'JAX version: {jax.__version__} | Backend: {jax.default_backend()}')
print('Devices:', jax.devices())

## 2. Numerical Parity Check: `build_block_lists_jax` & `pallas_block_list_attention`
Verifies the Pallas TPU online-softmax block-list kernel against an exact `fp32` masked attention reference across `p in {2, 3}`, `req_depth in {1, 2, 3, 4}`, causal masking, attention sink, and local window.

In [ ]:
import ultrametric_jax
import ultrametric_jax.kernel
importlib.reload(ultrametric_jax.kernel)
importlib.reload(ultrametric_jax)
from ultrametric_jax.kernel import build_block_lists_jax, pallas_block_list_attention

is_tpu = jax.default_backend() == 'tpu'
interpret_mode = not is_tpu  # Uses Pallas interpret mode automatically if testing on CPU/GPU

key = jax.random.PRNGKey(42)
Z, H, N, D, B, TD = 1, 8, 1024, 128, 128, 5
NB = N // B

kq, kk, kv, kr = jax.random.split(key, 4)
q = jax.random.normal(kq, (Z, H, N, D), dtype=jnp.float32)
k = jax.random.normal(kk, (Z, H, N, D), dtype=jnp.float32)
v = jax.random.normal(kv, (Z, H, N, D), dtype=jnp.float32)

for p in [2, 3]:
    routes = jax.random.randint(kr, (Z, H, NB, TD), 0, p, dtype=jnp.int32)
    for req_depth in [1, 2, 3, 4]:
        out_pallas = pallas_block_list_attention(
            q, k, v, routes, req_depth,
            block_size=B, is_causal=True, sink_block=True, local_blocks=1,
            interpret=interpret_mode
        )
        # Reference masked attention
        r_sub = routes[:, :, :, :req_depth]
        blk_match = jnp.all(r_sub[:, :, :, None, :] == r_sub[:, :, None, :, :], axis=-1)
        idx_b = jnp.arange(NB)
        blk_match = blk_match | (jnp.abs(idx_b[:, None] - idx_b[None, :]) <= 1)
        blk_match = blk_match.at[..., :, 0].set(True)
        blk_match = blk_match & (idx_b[:, None] >= idx_b[None, :])
        tok_mask = jnp.repeat(jnp.repeat(blk_match, B, axis=-2), B, axis=-1)
        idx_t = jnp.arange(N)
        tok_mask = tok_mask & (idx_t[:, None] >= idx_t[None, :])
        scores = jnp.einsum('bhqd,bhkd->bhqk', q, k, precision=jax.lax.Precision.HIGHEST) / jnp.sqrt(D)
        scores = jnp.where(tok_mask, scores, -1e30)
        weights = jax.nn.softmax(scores, axis=-1)
        out_ref = jnp.einsum('bhqk,bhkd->bhqd', weights, v, precision=jax.lax.Precision.HIGHEST)
        max_err = float(jnp.max(jnp.abs(out_pallas - out_ref)))
        lists, counts = build_block_lists_jax(routes, req_depth, is_causal=True, sink_block=True, local_blocks=1)
        density = 100.0 * float(jnp.sum(counts)) / (Z * H * NB * (NB + 1) / 2)
        print(f'p={p} | req_depth={req_depth} | ActiveCausalBlocks={density:6.2f}% | MaxAbsErr={max_err:.2e}')
        assert max_err < 1.5e-3, f'Parity error too large: {max_err}'

## 3. Track 4 Benchmark: Pallas Block-List Attention vs. Dense XLA Attention ($N \in [2048 .. 32768]$)
Measures wall-clock latency and effective HBM bandwidth scaling across sequence lengths on TPU v5e / v6e.

In [ ]:
import time

dtype = jnp.bfloat16 if is_tpu else jnp.float32
Z, H, D, B, TD = 1, 16, 128, 128, 6
req_depth = 4
seq_lens = [2048, 4096, 8192, 16384, 32768] if is_tpu else [1024, 2048]

@jax.jit
def dense_causal_attention(q, k, v):
    N = q.shape[2]
    scores = jnp.einsum('bhqd,bhkd->bhqk', q.astype(jnp.float32), k.astype(jnp.float32)) / jnp.sqrt(q.shape[-1])
    idx = jnp.arange(N)
    scores = jnp.where(idx[:, None] >= idx[None, :], scores, -1e30)
    w = jax.nn.softmax(scores, axis=-1).astype(v.dtype)
    return jnp.einsum('bhqk,bhkd->bhqd', w, v)

print(f'{"SeqLen":>8} | {"Active%":>8} | {"PallasSparse(ms)":>18} | {"DenseXLA(ms)":>14} | {"Speedup":>8}')
print('-' * 68)

for N in seq_lens:
    NB = N // B
    q = jax.random.normal(key, (Z, H, N, D), dtype=dtype)
    k = jax.random.normal(key, (Z, H, N, D), dtype=dtype)
    v = jax.random.normal(key, (Z, H, N, D), dtype=dtype)
    routes = jax.random.randint(key, (Z, H, NB, TD), 0, 2, dtype=jnp.int32)

    # Warmup / compile
    out_sp = pallas_block_list_attention(
        q, k, v, routes, req_depth, block_size=B, is_causal=True, sink_block=True, local_blocks=1, interpret=interpret_mode
    ).block_until_ready()

    t0 = time.perf_counter()
    iters = 10 if is_tpu else 2
    for _ in range(iters):
        out_sp = pallas_block_list_attention(
            q, k, v, routes, req_depth, block_size=B, is_causal=True, sink_block=True, local_blocks=1, interpret=interpret_mode
        ).block_until_ready()
    t_sp = (time.perf_counter() - t0) * 1000 / iters

    try:
        out_dn = dense_causal_attention(q, k, v).block_until_ready()
        t0 = time.perf_counter()
        for _ in range(iters):
            out_dn = dense_causal_attention(q, k, v).block_until_ready()
        t_dn = (time.perf_counter() - t0) * 1000 / iters
        speedup_str = f'{t_dn / t_sp:7.2f}x'
        t_dn_str = f'{t_dn:14.3f}'
    except Exception:
        t_dn_str = 'OOM'
        speedup_str = 'inf'

    _, counts = build_block_lists_jax(routes, req_depth, is_causal=True, sink_block=True, local_blocks=1)
    density = 100.0 * float(jnp.sum(counts)) / (Z * H * NB * (NB + 1) / 2)
    print(f'{N:8d} | {density:7.2f}% | {t_sp:18.3f} | {t_dn_str:>14} | {speedup_str:>8}')